In [ ]:

import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import yfinance as yf
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import feedparser
from bs4 import BeautifulSoup
import json
import time
import re
from transformers import pipeline
import warnings

warnings.filterwarnings('ignore')

print("🚀 실시간 뉴스 및 감성 분석 라이브러리 준비 완료!")


# 1. 다양한 뉴스 소스 크롤링 클래스
class SamsungNewsCollector:
    def __init__(self):
        self.analyzer = SentimentIntensityAnalyzer()
        self.sentiment_pipeline = pipeline("sentiment-analysis",
                                           model="nlptown/bert-base-multilingual-uncased-sentiment")

    def get_yahoo_finance_news(self, symbol="005930.KS", limit=50):
        """야후 파이낸스 뉴스 수집"""
        try:
            ticker = yf.Ticker(symbol)
            news = ticker.news

            news_data = []
            for article in news[:limit]:
                news_data.append({
                    'title': article.get('title', ''),
                    'summary': article.get('summary', ''),
                    'published': datetime.fromtimestamp(article.get('providerPublishTime', 0)),
                    'url': article.get('link', ''),
                    'source': 'Yahoo Finance'
                })

            return pd.DataFrame(news_data)
        except Exception as e:
            print(f"Yahoo Finance 뉴스 수집 오류: {e}")
            return pd.DataFrame()

    def get_google_finance_news(self, query="삼성전자", limit=50):
        """구글 파이낸스 RSS 뉴스 수집"""
        try:
            url = f"https://news.google.com/rss/search?q={query}&hl=ko&gl=KR&ceid=KR:ko"
            feed = feedparser.parse(url)

            news_data = []
            for entry in feed.entries[:limit]:
                news_data.append({
                    'title': entry.title,
                    'summary': getattr(entry, 'summary', ''),
                    'published': datetime(*entry.published_parsed[:6]),
                    'url': entry.link,
                    'source': 'Google News'
                })

            return pd.DataFrame(news_data)
        except Exception as e:
            print(f"Google 뉴스 수집 오류: {e}")
            return pd.DataFrame()

    def get_naver_finance_news(self, code="005930"):
        """네이버 파이낸스 뉴스 수집"""
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
            }

            url = f"https://finance.naver.com/item/news_news.naver?code={code}&page=1&sm=title_entity_id.basic&clusterId="
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.content, 'html.parser')

            news_data = []
            news_items = soup.find_all('tr', {'onmouseover': True})

            for item in news_items[:20]:  # 최대 20개
                try:
                    title_elem = item.find('a', {'class': 'tit'})
                    if title_elem:
                        title = title_elem.text.strip()
                        url = "https://finance.naver.com" + title_elem['href']

                        date_elem = item.find('td', {'class': 'date'})
                        if date_elem:
                            date_str = date_elem.text.strip()
                            # 간단한 날짜 파싱 (실제로는 더 정교해야 함)
                            published = datetime.now() - timedelta(days=1)

                        news_data.append({
                            'title': title,
                            'summary': '',
                            'published': published,
                            'url': url,
                            'source': 'Naver Finance'
                        })
                except Exception as e:
                    continue

            return pd.DataFrame(news_data)
        except Exception as e:
            print(f"네이버 뉴스 수집 오류: {e}")
            return pd.DataFrame()

    def analyze_sentiment(self, text):
        """다중 감성 분석"""
        if not text or pd.isna(text):
            return {'compound': 0, 'positive': 0, 'negative': 0, 'neutral': 0}

        # VADER 감성 분석
        vader_scores = self.analyzer.polarity_scores(text)

        # TextBlob 감성 분석
        blob = TextBlob(text)
        textblob_polarity = blob.sentiment.polarity

        # 종합 점수
        compound_score = (vader_scores['compound'] + textblob_polarity) / 2

        return {
            'compound': compound_score,
            'positive': vader_scores['pos'],
            'negative': vader_scores['neg'],
            'neutral': vader_scores['neu'],
            'textblob_polarity': textblob_polarity
        }

    def collect_all_news(self):
        """모든 소스에서 뉴스 수집"""
        print("📰 삼성전자 관련 뉴스 수집 시작...")

        all_news = []

        # Yahoo Finance 뉴스
        yahoo_news = self.get_yahoo_finance_news()
        if not yahoo_news.empty:
            all_news.append(yahoo_news)
            print(f"✅ Yahoo Finance: {len(yahoo_news)}개 뉴스 수집")

        # Google 뉴스
        google_news = self.get_google_finance_news()
        if not google_news.empty:
            all_news.append(google_news)
            print(f"✅ Google News: {len(google_news)}개 뉴스 수집")

        # 네이버 파이낸스 뉴스
        naver_news = self.get_naver_finance_news()
        if not naver_news.empty:
            all_news.append(naver_news)
            print(f"✅ Naver Finance: {len(naver_news)}개 뉴스 수집")

        if all_news:
            combined_news = pd.concat(all_news, ignore_index=True)

            # 중복 제거
            combined_news = combined_news.drop_duplicates(subset=['title'])

            # 감성 분석 수행
            print("🧠 감성 분석 수행 중...")
            sentiment_results = []
            for _, row in combined_news.iterrows():
                text = f"{row['title']} {row['summary']}"
                sentiment = self.analyze_sentiment(text)
                sentiment_results.append(sentiment)

            sentiment_df = pd.DataFrame(sentiment_results)
            combined_news = pd.concat([combined_news, sentiment_df], axis=1)

            print(f"✅ 총 {len(combined_news)}개 뉴스 수집 및 감성 분석 완료!")
            return combined_news

        return pd.DataFrame()


# 뉴스 수집기 초기화 및 실행
news_collector = SamsungNewsCollector()
current_news = news_collector.collect_all_news()


# 2. 경제 지표 및 시장 데이터 수집기
class EconomicIndicatorsCollector:
    def __init__(self):
        self.indicators = {}

    def get_korean_economic_indicators(self):
        """한국 경제 지표 수집"""
        try:
            # KOSPI 지수
            kospi = yf.Ticker("^KS11")
            kospi_data = kospi.history(period="1y")

            # 원달러 환율
            krw_usd = yf.Ticker("KRW=X")
            krw_data = krw_usd.history(period="1y")

            # 미국 금리 (10년 국채)
            us_10y = yf.Ticker("^TNX")
            us_10y_data = us_10y.history(period="1y")

            # VIX (공포 지수)
            vix = yf.Ticker("^VIX")
            vix_data = vix.history(period="1y")

            # 반도체 ETF (SMH)
            smh = yf.Ticker("SMH")
            smh_data = smh.history(period="1y")

            self.indicators = {
                'KOSPI': kospi_data['Close'],
                'KRW_USD': krw_data['Close'],
                'US_10Y': us_10y_data['Close'],
                'VIX': vix_data['Close'],
                'SMH': smh_data['Close']
            }

            print("✅ 경제 지표 수집 완료!")
            return self.indicators

        except Exception as e:
            print(f"경제 지표 수집 오류: {e}")
            return {}

    def get_sector_performance(self):
        """섹터별 성과 지표"""
        try:
            # 기술주 ETF들
            tech_etfs = {
                'QQQ': '^NDX',  # 나스닥
                'XLK': 'XLK',  # 기술 섹터 ETF
                'SOXX': 'SOXX',  # 반도체 ETF
            }

            sector_data = {}
            for name, symbol in tech_etfs.items():
                ticker = yf.Ticker(symbol)
                data = ticker.history(period="1y")
                sector_data[name] = data['Close']

            return sector_data

        except Exception as e:
            print(f"섹터 성과 수집 오류: {e}")
            return {}


# 경제 지표 수집
econ_collector = EconomicIndicatorsCollector()
economic_indicators = econ_collector.get_korean_economic_indicators()
sector_performance = econ_collector.get_sector_performance()


# 3. 뉴스와 주가 데이터 시간별 매칭
def match_news_to_price_data(news_df, price_data):
    """뉴스 데이터와 주가 데이터를 시간별로 매칭"""
    if news_df.empty:
        return price_data

    # 뉴스 데이터를 날짜별로 그룹화
    news_df['date'] = news_df['published'].dt.date
    daily_sentiment = news_df.groupby('date').agg({
        'compound': 'mean',
        'positive': 'mean',
        'negative': 'mean',
        'neutral': 'mean',
        'textblob_polarity': 'mean'
    }).reset_index()

    daily_sentiment['date'] = pd.to_datetime(daily_sentiment['date'])
    daily_sentiment.set_index('date', inplace=True)

    # 뉴스 개수도 추가
    news_count = news_df.groupby('date').size().reset_index(name='news_count')
    news_count['date'] = pd.to_datetime(news_count['date'])
    news_count.set_index('date', inplace=True)

    # 주가 데이터와 병합
    price_data_copy = price_data.copy()
    price_data_copy = price_data_copy.merge(daily_sentiment, left_index=True, right_index=True, how='left')
    price_data_copy = price_data_copy.merge(news_count, left_index=True, right_index=True, how='left')

    # 결측치 보간
    sentiment_columns = ['compound', 'positive', 'negative', 'neutral', 'textblob_polarity']
    for col in sentiment_columns:
        price_data_copy[col] = price_data_copy[col].fillna(method='ffill').fillna(0)

    price_data_copy['news_count'] = price_data_copy['news_count'].fillna(0)

    print(f"✅ 뉴스-주가 데이터 매칭 완료! {len(price_data_copy)}일의 데이터")
    return price_data_copy


# 뉴스와 주가 데이터 매칭
enhanced_price_data = match_news_to_price_data(current_news, enhanced_data)


# 4. 경제 지표와 주가 데이터 통합
def integrate_economic_indicators(price_data, economic_indicators, sector_performance):
    """경제 지표를 주가 데이터에 통합"""
    integrated_data = price_data.copy()

    # 경제 지표 추가
    for indicator_name, indicator_data in economic_indicators.items():
        # 인덱스를 날짜로 맞추기
        indicator_resampled = indicator_data.resample('D').last().fillna(method='ffill')
        integrated_data = integrated_data.merge(
            indicator_resampled.rename(f'econ_{indicator_name}'),
            left_index=True, right_index=True, how='left'
        )

    # 섹터 성과 지표 추가
    for sector_name, sector_data in sector_performance.items():
        sector_resampled = sector_data.resample('D').last().fillna(method='ffill')
        integrated_data = integrated_data.merge(
            sector_resampled.rename(f'sector_{sector_name}'),
            left_index=True, right_index=True, how='left'
        )

    # 경제 지표 기반 파생 변수 생성
    if 'econ_KOSPI' in integrated_data.columns:
        integrated_data['kospi_correlation'] = integrated_data['Close'].rolling(20).corr(integrated_data['econ_KOSPI'])
        integrated_data['kospi_ratio'] = integrated_data['Close'] / integrated_data['econ_KOSPI']

    if 'econ_KRW_USD' in integrated_data.columns:
        integrated_data['currency_impact'] = integrated_data['econ_KRW_USD'].pct_change() * -1  # 원화 강세 시 수출 부정적

    if 'econ_VIX' in integrated_data.columns:
        integrated_data['market_fear'] = integrated_data['econ_VIX']
        integrated_data['fear_level'] = pd.cut(integrated_data['econ_VIX'],
                                               bins=[0, 15, 25, 35, 100],
                                               labels=[0, 1, 2, 3])  # 공포 수준 레벨

    # 결측치 처리
    numeric_columns = integrated_data.select_dtypes(include=[np.number]).columns
    integrated_data[numeric_columns] = integrated_data[numeric_columns].fillna(method='ffill').fillna(method='bfill')

    print(f"✅ 경제 지표 통합 완료! 총 {len(integrated_data.columns)}개 변수")
    return integrated_data


# 경제 지표 통합
final_integrated_data = integrate_economic_indicators(enhanced_price_data, economic_indicators, sector_performance)

print(f"📊 최종 통합 데이터 컬럼: {list(final_integrated_data.columns)}")


# 5. 고급 피처 엔지니어링 (뉴스 + 경제지표 기반)
def create_advanced_news_features(data):
    """뉴스 기반 고급 피처 생성"""
    df = data.copy()

    # 감성 점수 기반 피처
    if 'compound' in df.columns:
        df['sentiment_momentum'] = df['compound'].rolling(5).mean()
        df['sentiment_volatility'] = df['compound'].rolling(10).std()
        df['sentiment_trend'] = df['compound'].rolling(3).apply(lambda x: 1 if x.iloc[-1] > x.iloc[0] else 0)

        # 극단적 감성 플래그
        df['extreme_positive'] = (df['compound'] > 0.5).astype(int)
        df['extreme_negative'] = (df['compound'] < -0.5).astype(int)

    # 뉴스 활동성 피처
    if 'news_count' in df.columns:
        df['news_activity'] = df['news_count'].rolling(7).sum()  # 주간 뉴스 활동
        df['news_spike'] = (df['news_count'] > df['news_count'].rolling(30).quantile(0.8)).astype(int)

    # 경제 지표 기반 피처
    if 'econ_KOSPI' in df.columns:
        df['market_divergence'] = (df['Close'].pct_change() - df['econ_KOSPI'].pct_change()).abs()
        df['outperforming_market'] = (df['Close'].pct_change() > df['econ_KOSPI'].pct_change()).astype(int)

    # 복합 신호 생성
    if all(col in df.columns for col in ['compound', 'econ_VIX', 'news_count']):
        # 시장 스트레스 지수
        df['market_stress'] = (
                (df['econ_VIX'] > 25).astype(int) * 0.4 +
                (df['compound'] < -0.3).astype(int) * 0.3 +
                (df['news_count'] > df['news_count'].quantile(0.8)).astype(int) * 0.3
        )

        # 투자 신호 강도
        df['investment_signal'] = (
                df['compound'] * 0.3 +
                (-df['econ_VIX'] / 100) * 0.3 +  # VIX는 역방향
                (df['news_count'] / df['news_count'].max()) * 0.2 +
                df['outperforming_market'] * 0.2
        )

    return df


# 고급 피처 생성
final_integrated_data = create_advanced_news_features(final_integrated_data)

# 최종 데이터 정리
clean_final_data = final_integrated_data.dropna()
print(f"✅ 최종 정제된 데이터: {clean_final_data.shape}")
print(f"기간: {clean_final_data.index.min()} ~ {clean_final_data.index.max()}")

# 6. 고도화된 예측 모델 (뉴스 + 경제지표 통합)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
import xgboost as xgb
import lightgbm as lgb


class EnhancedPredictionModel:
    def __init__(self):
        self.models = {}
        self.feature_importance = {}
        self.scaler = StandardScaler()

    def prepare_features(self, data, target_days=5):
        """통합 데이터에서 피처 준비"""
        # 타겟 변수 생성
        data[f'future_return_{target_days}d'] = data['Close'].pct_change(target_days).shift(-target_days)
        data[f'future_direction_{target_days}d'] = (data[f'future_return_{target_days}d'] > 0).astype(int)

        # 피처 선택 (수치형만)
        feature_columns = [col for col in data.columns if data[col].dtype in ['float64', 'int64']]
        feature_columns = [col for col in feature_columns if not col.startswith('future_')]

        # 결측치 있는 행 제거
        clean_data = data[
            feature_columns + [f'future_return_{target_days}d', f'future_direction_{target_days}d']].dropna()

        return clean_data, feature_columns

    def train_enhanced_models(self, data, feature_columns, target_col):
        """향상된 모델들 훈련"""
        X = data[feature_columns]
        y = data[target_col]

        # 시계열 분할
        split_idx = int(len(X) * 0.8)
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

        # 피처 스케일링
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)

        # 다양한 모델 훈련
        models_config = {
            'RandomForest': RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42),
            'XGBoost': xgb.XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
            'LightGBM': lgb.LGBMRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, random_state=42),
            'GradientBoosting': GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                                          random_state=42),
            'NeuralNetwork': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500, random_state=42)
        }

        results = {}

        for model_name, model in models_config.items():
            print(f"🔥 {model_name} 모델 훈련 중...")

            if model_name == 'NeuralNetwork':
                model.fit(X_train_scaled, y_train)
                y_pred = model.predict(X_test_scaled)
            else:
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)

            # 성능 평가
            mse = mean_squared_error(y_test, y_pred)
            mae = mean_absolute_error(y_test, y_pred)
            r2 = r2_score(y_test, y_pred)

            results[model_name] = {
                'model': model,
                'mse': mse,
                'mae': mae,
                'r2': r2,
                'predictions': y_pred,
                'actual': y_test
            }

            # 피처 중요도 (가능한 경우)
            if hasattr(model, 'feature_importances_'):
                importance_df = pd.DataFrame({
                    'feature': feature_columns,
                    'importance': model.feature_importances_
                }).sort_values('importance', ascending=False)
                self.feature_importance[model_name] = importance_df

            print(f"✅ {model_name} - R²: {r2:.4f}, MAE: {mae:.4f}")

        self.models = results
        return results

    def create_ensemble_prediction(self, weights=None):
        """앙상블 예측 생성"""
        if not weights:
            # 성능 기반 가중치 계산
            r2_scores = {name: result['r2'] for name, result in self.models.items()}
            total_r2 = sum(r2_scores.values())
            weights = {name: r2 / total_r2 for name, r2 in r2_scores.items()}

        # 길이 맞추기
        min_length = min(len(result['predictions']) for result in self.models.values())

        ensemble_pred = np.zeros(min_length)
        for model_name, weight in weights.items():
            predictions = self.models[model_name]['predictions'][:min_length]
            ensemble_pred += weight * predictions

        return ensemble_pred, weights


# 향상된 예측 모델 훈련
enhanced_model = EnhancedPredictionModel()
model_data, feature_cols = enhanced_model.prepare_features(clean_final_data, target_days=5)

print(f"📊 모델 훈련용 데이터: {model_data.shape}")
print(f"피처 개수: {len(feature_cols)}")

# 모델 훈련
results = enhanced_model.train_enhanced_models(model_data, feature_cols, 'future_return_5d')

# 앙상블 예측
ensemble_pred, ensemble_weights = enhanced_model.create_ensemble_prediction()

print(f"\n🎯 앙상블 가중치:")
for model, weight in ensemble_weights.items():
    print(f"  {model}: {weight:.3f}")


# 7. 실시간 예측 시스템
class RealTimePredictionSystem:
    def __init__(self, enhanced_model, news_collector, econ_collector):
        self.enhanced_model = enhanced_model
        self.news_collector = news_collector
        self.econ_collector = econ_collector

    def get_latest_data(self):
        """최신 데이터 수집"""
        print("🔄 실시간 데이터 수집 중...")

        # 최신 뉴스 수집
        latest_news = self.news_collector.collect_all_news()

        # 최신 경제 지표 수집
        latest_econ = self.econ_collector.get_korean_economic_indicators()
        latest_sector = self.econ_collector.get_sector_performance()

        # 최신 삼성전자 주가
        samsung = yf.Ticker("005930.KS")
        latest_price = samsung.history(period="5d")

        return latest_news, latest_econ, latest_sector, latest_price

    def make_realtime_prediction(self):
        """실시간 예측 수행"""
        try:
            # 최신 데이터 수집
            latest_news, latest_econ, latest_sector, latest_price = self.get_latest_data()

            # 데이터 통합 (기존 함수 재사용)
            if not latest_news.empty:
                integrated_latest = match_news_to_price_data(latest_news, latest_price)
            else:
                integrated_latest = latest_price

            integrated_latest = integrate_economic_indicators(integrated_latest, latest_econ, latest_sector)
            integrated_latest = create_advanced_news_features(integrated_latest)

            # 마지막 행의 피처 추출
            latest_features = integrated_latest[feature_cols].iloc[-1:].fillna(method='ffill').fillna(0)

            # 각 모델로 예측
            predictions = {}
            for model_name, model_info in self.enhanced_model.models.items():
                model = model_info['model']

                if model_name == 'NeuralNetwork':
                    # 스케일링 적용
                    latest_scaled = self.enhanced_model.scaler.transform(latest_features)
                    pred = model.predict(latest_scaled)[0]
                else:
                    pred = model.predict(latest_features)[0]

                predictions[model_name] = pred

            # 앙상블 예측
            ensemble_pred = sum(pred * ensemble_weights[name] for name, pred in predictions.items())

            # 현재 주가
            current_price = latest_price['Close'].iloc[-1]

            # 예측 결과 정리
            prediction_results = {
                'current_price': current_price,
                'predicted_return_5d': ensemble_pred,
                'predicted_price_5d': current_price * (1 + ensemble_pred),
                'individual_predictions': predictions,
                'confidence_level': self._calculate_confidence(predictions),
                'market_sentiment': self._analyze_current_sentiment(latest_news, integrated_latest),
                'economic_outlook': self._analyze_economic_outlook(integrated_latest)
            }

            return prediction_results

        except Exception as e:
            print(f"실시간 예측 오류: {e}")
            return None

    def _calculate_confidence(self, predictions):
        """예측 신뢰도 계산"""
        pred_values = list(predictions.values())
        std_dev = np.std(pred_values)

        if std_dev < 0.02:  # 2% 미만 차이
            return "높음"
        elif std_dev < 0.05:  # 5% 미만 차이
            return "보통"
        else:
            return "낮음"

    def _analyze_current_sentiment(self, news_df, integrated_data):
        """현재 시장 감성 분석"""
        if news_df.empty:
            return "정보 없음"

        avg_sentiment = news_df['compound'].mean()
        news_count = len(news_df)

        if avg_sentiment > 0.3 and news_count > 5:
            return f"매우 긍정적 (감성점수: {avg_sentiment:.2f}, 뉴스: {news_count}개)"
        elif avg_sentiment > 0.1:
            return f"긍정적 (감성점수: {avg_sentiment:.2f}, 뉴스: {news_count}개)"
        elif avg_sentiment > -0.1:
            return f"중립적 (감성점수: {avg_sentiment:.2f}, 뉴스: {news_count}개)"
        elif avg_sentiment > -0.3:
            return f"부정적 (감성점수: {avg_sentiment:.2f}, 뉴스: {news_count}개)"
        else:
            return f"매우 부정적 (감성점수: {avg_sentiment:.2f}, 뉴스: {news_count}개)"

    def _analyze_economic_outlook(self, integrated_data):
        """경제 전망 분석"""
        latest_row = integrated_data.iloc[-1]

        outlook_factors = []

        # VIX 분석
        if 'econ_VIX' in integrated_data.columns:
            vix_value = latest_row['econ_VIX']
            if vix_value < 15:
                outlook_factors.append("시장 안정")
            elif vix_value < 25:
                outlook_factors.append("보통 변동성")
            else:
                outlook_factors.append("높은 불확실성")

        # KOSPI 상관관계
        if 'kospi_correlation' in integrated_data.columns:
            corr_value = latest_row['kospi_correlation']
            if corr_value > 0.7:
                outlook_factors.append("시장과 동조화")
            elif corr_value < 0.3:
                outlook_factors.append("독립적 움직임")

        # 환율 영향
        if 'currency_impact' in integrated_data.columns:
            currency_impact = latest_row['currency_impact']
            if abs(currency_impact) > 0.02:
                outlook_factors.append("환율 변동 영향")

        return " | ".join(outlook_factors) if outlook_factors else "중립적"


# 실시간 예측 시스템 초기화
realtime_system = RealTimePredictionSystem(enhanced_model, news_collector, econ_collector)

# 실시간 예측 수행
print("🔮 실시간 예측 수행 중...")
realtime_prediction = realtime_system.make_realtime_prediction()

# 8. 종합 결과 3D 시각화
if realtime_prediction:
    # 예측 결과 출력
    print("\n" + "=" * 60)
    print("🎯 삼성전자 실시간 종합 예측 결과")
    print("=" * 60)
    print(f"📊 현재 주가: {realtime_prediction['current_price']:,.0f}원")
    print(f"🔮 5일 후 예상 수익률: {realtime_prediction['predicted_return_5d'] * 100:+.2f}%")
    print(f"💰 5일 후 예상 주가: {realtime_prediction['predicted_price_5d']:,.0f}원")
    print(f"🎪 신뢰도: {realtime_prediction['confidence_level']}")
    print(f"😊 시장 감성: {realtime_prediction['market_sentiment']}")
    print(f"📈 경제 전망: {realtime_prediction['economic_outlook']}")

    print(f"\n📋 개별 모델 예측:")
    for model, pred in realtime_prediction['individual_predictions'].items():
        print(f"  {model}: {pred * 100:+.2f}%")

# 고급 3D 종합 시각화
fig_comprehensive = make_subplots(
    rows=2, cols=2,
    specs=[[{"type": "scatter3d"}, {"type": "scatter3d"}],
           [{"type": "scatter3d"}, {"type": "scatter3d"}]],
    subplot_titles=(
        '🧠 모델 성능 비교',
        '📰 뉴스 감성 vs 주가',
        '💹 경제지표 vs 주가',
        '🔮 실시간 예측 결과'
    ),
    vertical_spacing=0.08,
    horizontal_spacing=0.05
)

# 1. 모델 성능 비교 (1,1)
model_names = list(results.keys())
r2_scores = [results[name]['r2'] for name in model_names]
mae_scores = [results[name]['mae'] for name in model_names]
mse_scores = [results[name]['mse'] for name in model_names]

fig_comprehensive.add_trace(
    go.Scatter3d(
        x=model_names,
        y=r2_scores,
        z=mae_scores,
        mode='markers',
        marker=dict(
            size=[score * 100 for score in r2_scores],
            color=r2_scores,
            colorscale='Viridis',
            showscale=True
        ),
        name='모델 성능',
        text=[f'R²: {r2:.3f}<br>MAE: {mae:.4f}' for r2, mae in zip(r2_scores, mae_scores)],
        hovertemplate='%{x}<br>%{text}<extra></extra>'
    ), row=1, col=1
)

# 2. 뉴스 감성 vs 주가 (1,2)
if 'compound' in clean_final_data.columns:
    recent_data = clean_final_data.tail(100)
    fig_comprehensive.add_trace(
        go.Scatter3d(
            x=recent_data.index,
            y=recent_data['Close'],
            z=recent_data['compound'],
            mode='markers+lines',
            marker=dict(
                size=4,
                color=recent_data['compound'],
                colorscale='RdYlGn',
                showscale=False
            ),
            line=dict(color='blue', width=2),
            name='감성-주가',
            hovertemplate='날짜: %{x}<br>주가: %{y:,.0f}원<br>감성: %{z:.2f}<extra></extra>'
        ), row=1, col=2
    )

# 3. 경제지표 vs 주가 (2,1)
if 'econ_KOSPI' in clean_final_data.columns:
    fig_comprehensive.add_trace(
        go.Scatter3d(
            x=recent_data['Close'],
            y=recent_data['econ_KOSPI'],
            z=recent_data['econ_VIX'] if 'econ_VIX' in recent_data.columns else recent_data['Volume'],
            mode='markers',
            marker=dict(
                size=5,
                color=recent_data['Close'],
                colorscale='Plasma',
                showscale=False
            ),
            name='경제지표-주가',
            hovertemplate='삼성: %{x:,.0f}<br>KOSPI: %{y:,.0f}<br>VIX: %{z:.1f}<extra></extra>'
        ), row=2, col=1
    )

# 4. 실시간 예측 결과 (2,2)
if realtime_prediction:
    pred_models = list(realtime_prediction['individual_predictions'].keys())
    pred_values = [realtime_prediction['individual_predictions'][model] * 100 for model in pred_models]

    fig_comprehensive.add_trace(
        go.Scatter3d(
            x=pred_models,
            y=pred_values,
            z=[realtime_prediction['predicted_price_5d']] * len(pred_models),
            mode='markers',
            marker=dict(
                size=15,
                color=pred_values,
                colorscale='RdYlGn',
                showscale=False,
                line=dict(color='black', width=2)
            ),
            name='실시간 예측',
            hovertemplate='%{x}<br>수익률: %{y:.2f}%<br>예상가: %{z:,.0f}원<extra></extra>'
        ), row=2, col=2
    )

fig_comprehensive.update_layout(
    title='🚀 삼성전자 실시간 종합 예측 대시보드',
    height=1000,
    showlegend=False
)

fig_comprehensive.show()
